# SMRD - word-count arm and stacked arm (Colab, CPU is enough)

Two new arms on the same 5 folds as before:

* **lexical**: TF-IDF of the source tweet and of all words in the thread, logistic regression. No GPU needed.
* **stack**: weighted average of the lexical and BiGCN probabilities.

The validation fold is kept exactly as before: it is never trained on, and it alone picks the lexical model's
regularisation strength and the stack's weights. Each test fold is predicted once.

Use the Drive that holds your earlier cross-validation results. If the BiGCN folds are not on this Drive they are
trained again here (short; slower on CPU). **Run the last cell before switching or disconnecting the runtime.**

In [ ]:
#@title 1. Dependencies (GPU optional)
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the lexical-stack branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'lexical-stack'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (CPU, ~2 min) - stop here if anything fails
!python -m pytest tests/test_lexical_stack.py tests/test_cv.py tests/test_gwo.py tests/test_propagation.py -q

## twitter16

About a minute for the lexical arm.

In [ ]:
#@title 4a. twitter16: lexical arm, 5 folds
!python -m src.run_cv train --dataset twitter16 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: BiGCN folds (skipped if already on this Drive)
!python -m src.run_cv train --dataset twitter16 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4c. twitter16: stack lexical + BiGCN, then compare
!python -m src.run_cv stack --dataset twitter16 --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

A few minutes.

In [ ]:
#@title 5a. twitter15: lexical arm, 5 folds
!python -m src.run_cv train --dataset twitter15 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: BiGCN folds (skipped if already on this Drive)
!python -m src.run_cv train --dataset twitter15 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5c. twitter15: stack lexical + BiGCN, then compare
!python -m src.run_cv stack --dataset twitter15 --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme

Largest; the lexical arm takes several minutes.

In [ ]:
#@title 6a. pheme: lexical arm, 5 folds
!python -m src.run_cv train --dataset pheme --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: BiGCN folds (skipped if already on this Drive)
!python -m src.run_cv train --dataset pheme --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6c. pheme: stack lexical + BiGCN, then compare
!python -m src.run_cv stack --dataset pheme --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv compare --all-datasets --ckpt-root {CKPT_ROOT}
for p in glob.glob('results/cv/*.json'):
    shutil.copy(p, CKPT_ROOT)

In [ ]:
#@title 8. Flush Drive (run before switching/disconnecting the runtime)
drive.flush_and_unmount()
print('Drive flushed and unmounted')